# 04: Model Evaluation

## Objectives
- Load trained model and evaluate performance
- Generate detailed classification metrics
- Create confusion matrix and analyze errors
- Visualize model predictions on test set
- Analyze per-class performance

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_recall_fscore_support, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Configuration

In [ ]:
DATA_DIR = Path('./data')
FOOD10_DIR = DATA_DIR / 'food10'
MODELS_DIR = Path('./models')
REPORTS_DIR = Path('./reports')

# Create reports directory
REPORTS_DIR.mkdir(exist_ok=True)

SELECTED_CLASSES = [
    'pizza', 'sushi', 'hamburger', 'hot_dog', 'french_fries',
    'ice_cream', 'omelette', 'pancakes', 'ramen', 'steak'
]

CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(SELECTED_CLASSES)}
IDX_TO_CLASS = {idx: cls for cls, idx in CLASS_TO_IDX.items()}

IMG_SIZE = 224
BATCH_SIZE = 32

print(f"Number of classes: {len(SELECTED_CLASSES)}")
print(f"Model path: {MODELS_DIR / 'best_model.pth'}")

## Step 1: Load Model and Data

In [ ]:
# Import model architecture from training notebook
class FoodDataset(torch.utils.data.Dataset):
    def __init__(self, data_dir, split='train', transform=None):
        self.data_dir = Path(data_dir)
        self.split = split
        self.transform = transform
        
        self.images = []
        self.labels = []
        
        split_dir = self.data_dir / split
        
        for class_name in SELECTED_CLASSES:
            class_dir = split_dir / class_name
            if class_dir.exists():
                for img_path in class_dir.glob('*.jpg'):
                    self.images.append(img_path)
                    self.labels.append(CLASS_TO_IDX[class_name])
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

class FoodClassifier(nn.Module):
    def __init__(self, num_classes=10, pretrained=True):
        super(FoodClassifier, self).__init__()
        
        import torchvision.models as models
        self.backbone = models.resnet18(pretrained=pretrained)
        
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        return self.backbone(x)

def load_model(model_path, num_classes=10):
    """Load trained model"""
    model = FoodClassifier(num_classes=num_classes, pretrained=False)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()
    return model

def get_transforms(img_size=224):
    """Get validation transforms"""
    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return val_transform

# Load model
model = load_model(MODELS_DIR / 'best_model.pth')
print(f"Model loaded successfully!")

# Create test dataset and loader
val_transform = get_transforms(IMG_SIZE)
test_dataset = FoodDataset(FOOD10_DIR, split='test', transform=val_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Test dataset size: {len(test_dataset)}")
print(f"Test batches: {len(test_loader)}")

## Step 2: Generate Predictions

In [ ]:
def generate_predictions(model, test_loader, device):
    """Generate predictions for test set"""
    all_predictions = []
    all_labels = []
    all_probabilities = []
    
    model.eval()
    
    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(test_loader):
            data, target = data.to(device), target.to(device)
            
            outputs = model(data)
            probabilities = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(target.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
            
            if batch_idx % 10 == 0:
                print(f"Processed batch {batch_idx}/{len(test_loader)}")
    
    return np.array(all_predictions), np.array(all_labels), np.array(all_probabilities)

# Generate predictions
predictions, true_labels, probabilities = generate_predictions(model, test_loader, device)

print(f"\nGenerated {len(predictions)} predictions")
print(f"Unique predictions: {np.unique(predictions)}")
print(f"Unique true labels: {np.unique(true_labels)}")

## Step 3: Overall Performance Metrics

In [ ]:
def calculate_overall_metrics(predictions, true_labels, probabilities):
    """Calculate overall performance metrics"""
    
    # Basic metrics
    accuracy = accuracy_score(true_labels, predictions)
    precision, recall, f1, support = precision_recall_fscore_support(
        true_labels, predictions, average='weighted'
    )
    
    # Per-class metrics
    precision_per_class, recall_per_class, f1_per_class, support_per_class = precision_recall_fscore_support(
        true_labels, predictions, average=None
    )
    
    # Create metrics dataframe
    metrics_df = pd.DataFrame({
        'Class': SELECTED_CLASSES,
        'Precision': precision_per_class,
        'Recall': recall_per_class,
        'F1-Score': f1_per_class,
        'Support': support_per_class
    })
    
    # Add overall metrics
    overall_row = pd.DataFrame({
        'Class': ['Overall'],
        'Precision': [precision],
        'Recall': [recall],
        'F1-Score': [f1],
        'Support': [len(true_labels)]
    })
    
    metrics_df = pd.concat([metrics_df, overall_row], ignore_index=True)
    
    return metrics_df, accuracy

metrics_df, overall_accuracy = calculate_overall_metrics(predictions, true_labels, probabilities)

print("Overall Performance Metrics:")
print("=" * 60)
print(metrics_df.round(4))

print(f"\nOverall Accuracy: {overall_accuracy:.4f} ({overall_accuracy*100:.2f}%)")

## Step 4: Confusion Matrix

In [ ]:
def plot_confusion_matrix(predictions, true_labels, class_names):
    """Plot confusion matrix"""
    
    # Calculate confusion matrix
    cm = confusion_matrix(true_labels, predictions)
    
    # Create figure
    plt.figure(figsize=(12, 10))
    
    # Plot heatmap
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Count'})
    
    plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
    
    # Calculate per-class accuracy
    per_class_accuracy = cm.diagonal() / cm.sum(axis=1)
    
    print("\nPer-class Accuracy:")
    print("=" * 30)
    for i, class_name in enumerate(class_names):
        print(f"{class_name}: {per_class_accuracy[i]:.4f} ({per_class_accuracy[i]*100:.2f}%)")
    
    return cm, per_class_accuracy

cm, per_class_accuracy = plot_confusion_matrix(predictions, true_labels, SELECTED_CLASSES)

## Step 5: Classification Report

In [ ]:
def generate_classification_report(predictions, true_labels, class_names):
    """Generate detailed classification report"""
    
    report = classification_report(
        true_labels, predictions, 
        target_names=class_names, 
        digits=4,
        output_dict=True
    )
    
    # Convert to DataFrame for better visualization
    report_df = pd.DataFrame(report).transpose()
    
    print("Detailed Classification Report:")
    print("=" * 80)
    print(classification_report(true_labels, predictions, target_names=class_names, digits=4))
    
    return report_df

report_df = generate_classification_report(predictions, true_labels, SELECTED_CLASSES)

## Step 6: Error Analysis

In [ ]:
def analyze_errors(predictions, true_labels, probabilities, test_dataset, top_n=10):
    """Analyze prediction errors"""
    
    # Find incorrect predictions
    incorrect_mask = predictions != true_labels
    incorrect_indices = np.where(incorrect_mask)[0]
    
    print(f"Total errors: {len(incorrect_indices)} out of {len(true_labels)}")
    print(f"Error rate: {len(incorrect_indices)/len(true_labels)*100:.2f}%")
    
    # Calculate confidence for incorrect predictions
    incorrect_confidences = np.max(probabilities[incorrect_mask], axis=1)
    correct_confidences = np.max(probabilities[~incorrect_mask], axis=1)
    
    # Plot confidence distribution
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    axes[0].hist(correct_confidences, bins=30, alpha=0.7, label='Correct', color='green')
    axes[0].hist(incorrect_confidences, bins=30, alpha=0.7, label='Incorrect', color='red')
    axes[0].set_title('Prediction Confidence Distribution')
    axes[0].set_xlabel('Confidence')
    axes[0].set_ylabel('Frequency')
    axes[0].legend()
    
    # Most common error pairs
    error_pairs = []
    for idx in incorrect_indices:
        true_class = IDX_TO_CLASS[true_labels[idx]]
        pred_class = IDX_TO_CLASS[predictions[idx]]
        error_pairs.append(f"{true_class} → {pred_class}")
    
    from collections import Counter
    error_counts = Counter(error_pairs)
    top_errors = error_counts.most_common(top_n)
    
    # Plot top error pairs
    errors, counts = zip(*top_errors)
    axes[1].barh(range(len(errors)), counts)
    axes[1].set_yticks(range(len(errors)))
    axes[1].set_yticklabels(errors)
    axes[1].set_xlabel('Count')
    axes[1].set_title(f'Top {top_n} Error Types')
    axes[1].invert_yaxis()
    
    plt.tight_layout()
    plt.show()
    
    # Print top errors
    print(f"\nTop {top_n} Error Types:")
    print("=" * 40)
    for error, count in top_errors:
        print(f"{error}: {count}")
    
    return incorrect_indices, error_counts

incorrect_indices, error_counts = analyze_errors(predictions, true_labels, probabilities, test_dataset)

## Step 7: Visualize Correct and Incorrect Predictions

In [ ]:
def visualize_predictions(test_dataset, predictions, true_labels, probabilities, num_samples=8):
    """Visualize correct and incorrect predictions"""
    
    # Find correct and incorrect predictions
    correct_mask = predictions == true_labels
    incorrect_mask = ~correct_mask
    
    correct_indices = np.where(correct_mask)[0]
    incorrect_indices = np.where(incorrect_mask)[0]
    
    # Sample indices
    num_correct = min(num_samples // 2, len(correct_indices))
    num_incorrect = min(num_samples // 2, len(incorrect_indices))
    
    sampled_correct = np.random.choice(correct_indices, num_correct, replace=False) if num_correct > 0 else []
    sampled_incorrect = np.random.choice(incorrect_indices, num_incorrect, replace=False) if num_incorrect > 0 else []
    
    # Create figure
    fig, axes = plt.subplots(2, num_samples, figsize=(20, 8))
    fig.suptitle('Model Predictions Visualization', fontsize=16, fontweight='bold')
    
    # Plot correct predictions
    for i, idx in enumerate(sampled_correct):
        if i >= num_samples // 2:
            break
            
        img_path, true_label = test_dataset.images[idx], test_dataset.labels[idx]
        pred_label = predictions[idx]
        confidence = probabilities[idx][pred_label]
        
        # Load and display image
        img = Image.open(img_path)
        axes[0, i].imshow(img)
        axes[0, i].set_title(
            f"Correct\nTrue: {IDX_TO_CLASS[true_label]}\nPred: {IDX_TO_CLASS[pred_label]}\nConf: {confidence:.3f}",
            color='green', fontsize=10
        )
        axes[0, i].axis('off')
    
    # Fill remaining correct slots
    for i in range(len(sampled_correct), num_samples // 2):
        axes[0, i].axis('off')
    
    # Plot incorrect predictions
    for i, idx in enumerate(sampled_incorrect):
        if i >= num_samples // 2:
            break
            
        img_path, true_label = test_dataset.images[idx], test_dataset.labels[idx]
        pred_label = predictions[idx]
        confidence = probabilities[idx][pred_label]
        
        # Load and display image
        img = Image.open(img_path)
        axes[1, i].imshow(img)
        axes[1, i].set_title(
            f"Incorrect\nTrue: {IDX_TO_CLASS[true_label]}\nPred: {IDX_TO_CLASS[pred_label]}\nConf: {confidence:.3f}",
            color='red', fontsize=10
        )
        axes[1, i].axis('off')
    
    # Fill remaining incorrect slots
    for i in range(len(sampled_incorrect), num_samples // 2):
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.show()

visualize_predictions(test_dataset, predictions, true_labels, probabilities)

## Step 8: Per-Class Performance Analysis

In [ ]:
def per_class_performance_analysis(predictions, true_labels, probabilities):
    """Detailed per-class performance analysis"""
    
    # Calculate per-class metrics
    precision_per_class, recall_per_class, f1_per_class, support_per_class = precision_recall_fscore_support(
        true_labels, predictions, average=None
    )
    
    # Create comprehensive per-class dataframe
    per_class_data = []
    
    for i, class_name in enumerate(SELECTED_CLASSES):
        class_mask = true_labels == i
        class_predictions = predictions[class_mask]
        class_true = true_labels[class_mask]
        class_probs = probabilities[class_mask]
        
        # Calculate class-specific metrics
        class_accuracy = accuracy_score(class_true, class_predictions)
        avg_confidence = np.mean(np.max(class_probs, axis=1))
        
        per_class_data.append({
            'Class': class_name,
            'Accuracy': class_accuracy,
            'Precision': precision_per_class[i],
            'Recall': recall_per_class[i],
            'F1-Score': f1_per_class[i],
            'Support': support_per_class[i],
            'Avg_Confidence': avg_confidence
        })
    
    per_class_df = pd.DataFrame(per_class_data)
    
    # Sort by F1-score
    per_class_df = per_class_df.sort_values('F1-Score', ascending=False)
    
    # Create visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Per-Class Performance Analysis', fontsize=16, fontweight='bold')
    
    # Bar plot of metrics
    x_pos = np.arange(len(SELECTED_CLASSES))
    width = 0.2
    
    axes[0, 0].bar(x_pos - width, per_class_df['Precision'], width, label='Precision', alpha=0.8)
    axes[0, 0].bar(x_pos, per_class_df['Recall'], width, label='Recall', alpha=0.8)
    axes[0, 0].bar(x_pos + width, per_class_df['F1-Score'], width, label='F1-Score', alpha=0.8)
    
    axes[0, 0].set_xlabel('Food Class')
    axes[0, 0].set_ylabel('Score')
    axes[0, 0].set_title('Precision, Recall, and F1-Score by Class')
    axes[0, 0].set_xticks(x_pos)
    axes[0, 0].set_xticklabels(per_class_df['Class'], rotation=45)
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Support distribution
    axes[0, 1].bar(per_class_df['Class'], per_class_df['Support'], color='skyblue', alpha=0.7)
    axes[0, 1].set_xlabel('Food Class')
    axes[0, 1].set_ylabel('Number of Samples')
    axes[0, 1].set_title('Test Set Support by Class')
    axes[0, 1].tick_params(axis='x', rotation=45)
    axes[0, 1].grid(True, alpha=0.3)
    
    # Accuracy vs Confidence scatter
    axes[1, 0].scatter(per_class_df['Avg_Confidence'], per_class_df['Accuracy'], 
                      s=100, alpha=0.7, c=range(len(per_class_df)), cmap='viridis')
    axes[1, 0].set_xlabel('Average Confidence')
    axes[1, 0].set_ylabel('Accuracy')
    axes[1, 0].set_title('Accuracy vs Average Confidence')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Add class labels to scatter plot
    for i, row in per_class_df.iterrows():
        axes[1, 0].annotate(row['Class'], (row['Avg_Confidence'], row['Accuracy']), 
                           xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    # Performance ranking table
    axes[1, 1].axis('off')
    table_data = per_class_df[['Class', 'F1-Score', 'Accuracy']].round(3)
    table = axes[1, 1].table(cellText=table_data.values, 
                             colLabels=table_data.columns,
                             cellLoc='center',
                             loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.5)
    axes[1, 1].set_title('Performance Ranking (by F1-Score)', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return per_class_df

per_class_df = per_class_performance_analysis(predictions, true_labels, probabilities)

## Step 9: Save Evaluation Report

In [ ]:
def save_evaluation_report(metrics_df, per_class_df, report_df, cm):
    """Save comprehensive evaluation report"""
    
    # Create timestamp
    from datetime import datetime
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Save metrics to CSV
    metrics_df.to_csv(REPORTS_DIR / f'metrics_{timestamp}.csv', index=False)
    per_class_df.to_csv(REPORTS_DIR / f'per_class_metrics_{timestamp}.csv', index=False)
    
    # Save confusion matrix
    cm_df = pd.DataFrame(cm, index=SELECTED_CLASSES, columns=SELECTED_CLASSES)
    cm_df.to_csv(REPORTS_DIR / f'confusion_matrix_{timestamp}.csv')
    
    # Create text report
    report_text = f"""
Food Classification Model Evaluation Report
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
===============================================

OVERALL PERFORMANCE:
- Accuracy: {overall_accuracy:.4f} ({overall_accuracy*100:.2f}%)
- Total Test Samples: {len(true_labels)}
- Number of Classes: {len(SELECTED_CLASSES)}

PER-CLASS PERFORMANCE (Top 5 by F1-Score):
{per_class_df.head(5)[['Class', 'F1-Score', 'Accuracy', 'Precision', 'Recall']].to_string(index=False)}

PER-CLASS PERFORMANCE (Bottom 5 by F1-Score):
{per_class_df.tail(5)[['Class', 'F1-Score', 'Accuracy', 'Precision', 'Recall']].to_string(index=False)}

COMMON ERRORS:
{error_counts.most_common(5) if 'error_counts' in locals() else 'N/A'}

RECOMMENDATIONS:
1. Focus on classes with low F1-scores for improvement
2. Consider data augmentation for underperforming classes
3. Analyze common error patterns for targeted improvements
"""
    
    with open(REPORTS_DIR / f'evaluation_report_{timestamp}.txt', 'w') as f:
        f.write(report_text)
    
    print(f"Evaluation report saved to {REPORTS_DIR}")
    print(f"Files created:")
    print(f"- metrics_{timestamp}.csv")
    print(f"- per_class_metrics_{timestamp}.csv")
    print(f"- confusion_matrix_{timestamp}.csv")
    print(f"- evaluation_report_{timestamp}.txt")

save_evaluation_report(metrics_df, per_class_df, report_df, cm)

## Summary

In [ ]:
print("\n" + "=" * 50)
print("MODEL EVALUATION SUMMARY")
print("=" * 50)
print(f"✅ Model loaded and evaluated on test set")
print(f"✅ Overall accuracy: {overall_accuracy:.2f}%")
print(f"✅ Confusion matrix generated")
print(f"✅ Detailed classification report created")
print(f"✅ Error analysis performed")
print(f"✅ Per-class performance analyzed")
print(f"✅ Evaluation report saved to {REPORTS_DIR}")

print(f"\nKey Findings:")
best_class = per_class_df.iloc[0]['Class']
worst_class = per_class_df.iloc[-1]['Class']
print(f"- Best performing class: {best_class} (F1: {per_class_df.iloc[0]['F1-Score']:.3f})")
print(f"- Worst performing class: {worst_class} (F1: {per_class_df.iloc[-1]['F1-Score']:.3f})")
print(f"- Average confidence: {per_class_df['Avg_Confidence'].mean():.3f}")

print("\nNext steps:")
print("1. Run 05_Inference.ipynb for testing on new images")
print("2. Consider model improvements based on error analysis")
print("3. Deploy model using FastAPI (see src/api)")